In [1]:
# =====================================================
# SanketSetu Transformer V5.2 - Imports & Dataset
# =====================================================

import os
import random
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print("TensorFlow Version:", tf.__version__)

# ---------------- Dataset ----------------

DATA_PATH = "../dataset/training"

X_train = np.load(os.path.join(DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(DATA_PATH, "y_train.npy"))

X_val = np.load(os.path.join(DATA_PATH, "X_val.npy"))
y_val = np.load(os.path.join(DATA_PATH, "y_val.npy"))

X_test = np.load(os.path.join(DATA_PATH, "X_test.npy"))
y_test = np.load(os.path.join(DATA_PATH, "y_test.npy"))

print("✅ Balanced Dataset Loaded!")
print("Train :", X_train.shape, y_train.shape)
print("Val   :", X_val.shape, y_val.shape)
print("Test  :", X_test.shape, y_test.shape)

num_classes = len(np.unique(y_train))
sequence_length = X_train.shape[1]
feature_dim = X_train.shape[2]

print("Classes :", num_classes)
print("Sequence Length :", sequence_length)
print("Features :", feature_dim)

TensorFlow Version: 2.21.0
✅ Balanced Dataset Loaded!
Train : (7320, 154, 63) (7320,)
Val   : (630, 154, 63) (630,)
Test  : (420, 154, 63) (420,)
Classes : 210
Sequence Length : 154
Features : 63


In [5]:
# ===============================================
# Transformer V5.1 - Gentle Augmentation Function
# ===============================================

def augment_sequence(sequence):
    seq = sequence.copy()

    # Pick ONE augmentation randomly
    choice = np.random.choice(["rotate", "scale", "translate", "noise"])

    if choice == "rotate":
        angle = np.deg2rad(np.random.uniform(-5, 5))
        c, s = np.cos(angle), np.sin(angle)

        for i in range(21):
            x = seq[:, i * 3]
            y = seq[:, i * 3 + 1]

            seq[:, i * 3] = x * c - y * s
            seq[:, i * 3 + 1] = x * s + y * c

    elif choice == "scale":
        scale = np.random.uniform(0.98, 1.02)
        seq *= scale

    elif choice == "translate":
        tx = np.random.uniform(-0.01, 0.01)
        ty = np.random.uniform(-0.01, 0.01)

        for i in range(21):
            seq[:, i * 3] += tx
            seq[:, i * 3 + 1] += ty

    elif choice == "noise":
        seq += np.random.normal(0, 0.0015, seq.shape)

    return seq

print("✅ Gentle augmentation function ready!")

✅ Gentle augmentation function ready!


In [10]:
# ============================================
# Reload ORIGINAL training dataset
# ============================================

DATA_PATH = "../dataset/training"

X_train = np.load(os.path.join(DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(DATA_PATH, "y_train.npy"))

print("✅ Original dataset restored!")
print("Train shape:", X_train.shape)
print("Labels shape:", y_train.shape)

✅ Original dataset restored!
Train shape: (7320, 154, 63)
Labels shape: (7320,)


In [11]:
# ============================================
# Transformer V5.1 — Controlled Augmentation
# ============================================

augmented_X = []
augmented_y = []

augmentation_ratio = 0.35   # Augment only 35% of training samples

for seq, label in zip(X_train, y_train):

    # Keep original sample
    augmented_X.append(seq)
    augmented_y.append(label)

    # Augment only some samples
    if np.random.rand() < augmentation_ratio:
        augmented_X.append(augment_sequence(seq))
        augmented_y.append(label)

# Convert to NumPy arrays
X_train = np.array(augmented_X, dtype=np.float32)
y_train = np.array(augmented_y)

# Shuffle dataset
perm = np.random.permutation(len(X_train))
X_train = X_train[perm]
y_train = y_train[perm]

print("✅ Controlled augmentation complete!")
print("Augmentation Ratio :", augmentation_ratio)
print("New Train Shape     :", X_train.shape)
print("Labels Shape        :", y_train.shape)

✅ Controlled augmentation complete!
Augmentation Ratio : 0.35
New Train Shape     : (9948, 154, 63)
Labels Shape        : (9948,)


In [12]:
# ============================================
# Transformer V5.1 - Normalize Dataset
# ============================================

train_mean = X_train.mean(axis=(0, 1), keepdims=True)
train_std = X_train.std(axis=(0, 1), keepdims=True) + 1e-8

X_train = (X_train - train_mean) / train_std
X_val = (X_val - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

print("✅ Normalization complete!")

print("Train mean :", round(float(X_train.mean()), 6))
print("Train std  :", round(float(X_train.std()), 6))
print("Val mean   :", round(float(X_val.mean()), 6))
print("Test mean  :", round(float(X_test.mean()), 6))

✅ Normalization complete!
Train mean : -2e-06
Train std  : 0.99977
Val mean   : -0.291092
Test mean  : -0.240318


In [13]:
# ============================================
# Transformer V5.1 - Fresh Model
# ============================================

tf.keras.backend.clear_session()

def transformer_block(inputs, head_size=16, num_heads=8, ff_dim=256, dropout=0.3):

    x = layers.MultiHeadAttention(
        key_dim=head_size,
        num_heads=num_heads,
        dropout=dropout
    )(inputs, inputs)

    x = layers.Add()([inputs, x])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    y = layers.Dense(ff_dim, activation="gelu")(x)
    y = layers.Dropout(dropout)(y)
    y = layers.Dense(inputs.shape[-1])(y)

    x = layers.Add()([x, y])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    return x


inputs = layers.Input(shape=(sequence_length, feature_dim))

x = layers.Dense(128)(inputs)

# Two Transformer blocks
x = transformer_block(x)
x = transformer_block(x)

x = layers.GlobalAveragePooling1D()(x)

x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="gelu")(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(num_classes, activation="softmax")(x)

model = Model(inputs, outputs)

print("✅ Fresh Transformer V5.1 model created!")
print("Output shape:", model.output_shape)

✅ Fresh Transformer V5.1 model created!
Output shape: (None, 210)


In [14]:
print("Training samples :", len(X_train))
print("Validation samples:", len(X_val))
print("Batch size:", 32)
print("Expected steps per epoch:", len(X_train)//32)

Training samples : 9948
Validation samples: 630
Batch size: 32
Expected steps per epoch: 310


In [ ]:
# ============================================
# Transformer V5.1 - Compile + Train
# ============================================

# Cosine Decay Learning Rate
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=5e-4,
    decay_steps=60 * (len(X_train) // 32),
    alpha=0.1
)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=15,
    restore_best_weights=True,
    mode="max",
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "../models/best_transformer_v5_2.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

print("✅ Transformer V5.1 training complete!")

Epoch 1/60
311/311 ━━━━━━━━━━━━━━━━━━━━ 0s 443ms/step - accuracy: 0.0195 - loss: 5.1049
Epoch 1: val_accuracy improved from None to 0.00635, saving model to ../models/best_transformer_v5_2.keras

Epoch 1: finished saving model to ../models/best_transformer_v5_2.keras
311/311 ━━━━━━━━━━━━━━━━━━━━ 146s 452ms/step - accuracy: 0.0195 - loss: 5.1049 - val_accuracy: 0.0063 - val_loss: 5.2618
Epoch 2/60
311/311 ━━━━━━━━━━━━━━━━━━━━ 0s 407ms/step - accuracy: 0.0817 - loss: 4.2675
Epoch 2: val_accuracy improved from 0.00635 to 0.01587, saving model to ../models/best_transformer_v5_2.keras

Epoch 2: finished saving model to ../models/best_transformer_v5_2.keras
311/311 ━━━━━━━━━━━━━━━━━━━━ 129s 416ms/step - accuracy: 0.0817 - loss: 4.2675 - val_accuracy: 0.0159 - val_loss: 5.5981
Epoch 3/60
 35/311 ━━━━━━━━━━━━━━━━━━━━ 1:48 392ms/step - accuracy: 0.1429 - loss: 3.7779